# Análise da Tabela 3 — Áreas gerais de formação na graduação

**Fonte:** IBGE, Censo Demográfico — Tabela 10063 (pessoas com nível superior completo, por área geral de formação do curso de graduação concluído, segundo o sexo).

Neste notebook a tabela original é dividida em 3 tabelas por **Grande Região**:

1. Homens e mulheres com curso superior (total)
2. Homens e mulheres com curso em **Ciência, Tecnologia, Engenharias e Matemática**
3. Homens e mulheres com curso em **Educação, Serviços pessoais, Saúde e Bem-estar**

Cada tabela é salva em `.xlsx` na mesma pasta do notebook e analisada com `describe()`.

## 1. Importações e configurações

In [1]:
from pathlib import Path
import pandas as pd

# Mostra os números com separador de milhar e 2 casas decimais
pd.options.display.float_format = "{:,.2f}".format

# Pasta onde o notebook está (e onde a tabela e os .xlsx ficam)
PASTA = Path.cwd()
ARQUIVO = PASTA / "Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx"

print("Pasta de trabalho:", PASTA)
print("Arquivo existe?  ", ARQUIVO.exists())

Pasta de trabalho: /mnt/user-data/outputs
Arquivo existe?   True


## 2. Leitura da tabela original

O arquivo tem 3 linhas de cabeçalho (título, grupos e Total/Homens/Mulheres), por isso pulamos essas linhas e nomeamos as colunas manualmente.

In [2]:
bruta = pd.read_excel(ARQUIVO, header=None, skiprows=3)

bruta.columns = [
    "Local",
    "Superior_Total", "Superior_Homens", "Superior_Mulheres",
    "CTEM_Total", "CTEM_Homens", "CTEM_Mulheres",
    "Educ_Saude_Total", "Educ_Saude_Homens", "Educ_Saude_Mulheres",
]

bruta.head(10)

          Local  Superior_Total  Superior_Homens  Superior_Mulheres   CTEM_Total  CTEM_Homens  CTEM_Mulheres  Educ_Saude_Total  Educ_Saude_Homens  Educ_Saude_Mulheres
0        Brasil   25,854,291.00    10,478,534.00      15,375,757.00 4,149,041.00 2,750,422.00   1,398,619.00      8,079,707.00       1,677,985.00         6,401,723.00
1         Norte    1,525,166.00       590,558.00         934,609.00   209,657.00   133,055.00      76,601.00        599,785.00         141,169.00           458,616.00
2      Nordeste    4,748,933.00     1,744,994.00       3,003,939.00   621,388.00   389,993.00     231,395.00      1,822,087.00         363,905.00         1,458,181.00
3       Sudeste   12,828,829.00     5,391,638.00       7,437,191.00 2,328,122.00 1,588,874.00     739,247.00      3,652,397.00         768,288.00         2,884,108.00
4           Sul    4,338,861.00     1,768,855.00       2,570,005.00   662,916.00   427,372.00     235,543.00      1,256,291.00         248,574.00         1,007,718.0

## 3. Selecionando as Grandes Regiões

A tabela traz Brasil, Grandes Regiões e Unidades da Federação. Como a análise é **por região**, mantemos apenas as 5 Grandes Regiões (o total do Brasil é usado só para conferência).

In [3]:
REGIOES = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

regioes = bruta[bruta["Local"].isin(REGIOES)].set_index("Local")
regioes.index.name = "Região"
regioes = regioes.astype("int64")

regioes

              Superior_Total  Superior_Homens  Superior_Mulheres  CTEM_Total  CTEM_Homens  CTEM_Mulheres  Educ_Saude_Total  Educ_Saude_Homens  Educ_Saude_Mulheres
Região                                                                                                                                                            
Norte                1525166           590558             934609      209657       133055          76601            599785             141169               458616
Nordeste             4748933          1744994            3003939      621388       389993         231395           1822087             363905              1458181
Sudeste             12828829          5391638            7437191     2328122      1588874         739247           3652397             768288              2884108
Sul                  4338861          1768855            2570005      662916       427372         235543           1256291             248574              1007718
Centro-Oeste         2

In [4]:
# Conferência: a soma das 5 regiões deve bater com o total do Brasil
brasil = bruta[bruta["Local"] == "Brasil"].set_index("Local").iloc[0]
print("Diferença (soma das regiões - Brasil) por coluna:")
print((regioes.sum() - brasil).to_string())

Diferença (soma das regiões - Brasil) por coluna:
Superior_Total         0.00
Superior_Homens        0.00
Superior_Mulheres      0.00
CTEM_Total             1.00
CTEM_Homens           -4.00
CTEM_Mulheres          1.00
Educ_Saude_Total       1.00
Educ_Saude_Homens     -1.00
Educ_Saude_Mulheres    1.00


## 4. Dividindo a tabela em 3

In [5]:
# 1) Homens e mulheres por região com curso superior
superior = regioes[["Superior_Homens", "Superior_Mulheres"]].copy()
superior.columns = ["Homens", "Mulheres"]

# 2) Homens e mulheres por região — Ciência, Tecnologia, Engenharias e Matemática
tecnologia = regioes[["CTEM_Homens", "CTEM_Mulheres"]].copy()
tecnologia.columns = ["Homens", "Mulheres"]

# 3) Homens e mulheres por região — Educação, Serviços pessoais, Saúde e Bem-estar
educacao_saude = regioes[["Educ_Saude_Homens", "Educ_Saude_Mulheres"]].copy()
educacao_saude.columns = ["Homens", "Mulheres"]

print("Homens e mulheres com curso superior, por região")
display(superior)
print("Homens e mulheres em Ciência, Tecnologia, Engenharias e Matemática, por região")
display(tecnologia)
print("Homens e mulheres em Educação, Serviços pessoais, Saúde e Bem-estar, por região")
display(educacao_saude)

Homens e mulheres com curso superior, por região
               Homens  Mulheres
Região                         
Norte          590558    934609
Nordeste      1744994   3003939
Sudeste       5391638   7437191
Sul           1768855   2570005
Centro-Oeste   982489   1430013
Homens e mulheres em Ciência, Tecnologia, Engenharias e Matemática, por região
               Homens  Mulheres
Região                         
Norte          133055     76601
Nordeste       389993    231395
Sudeste       1588874    739247
Sul            427372    235543
Centro-Oeste   211124    115834
Homens e mulheres em Educação, Serviços pessoais, Saúde e Bem-estar, por região
              Homens  Mulheres
Região                        
Norte         141169    458616
Nordeste      363905   1458181
Sudeste       768288   2884108
Sul           248574   1007718
Centro-Oeste  156048    593101


## 5. Salvando as tabelas em `.xlsx`

Usamos `Path` para montar o caminho (evita problemas com `\` no Windows, onde `"\t"` seria interpretado como uma tabulação).

In [6]:
superior.to_excel(PASTA / "ensino_superior.xlsx")
tecnologia.to_excel(PASTA / "tecnologia.xlsx")
educacao_saude.to_excel(PASTA / "educacao_saude.xlsx")

print("Arquivos salvos em:", PASTA)
for nome in ["ensino_superior.xlsx", "tecnologia.xlsx", "educacao_saude.xlsx"]:
    print(" -", nome, "->", (PASTA / nome).exists())

Arquivos salvos em: /mnt/user-data/outputs
 - ensino_superior.xlsx -> True
 - tecnologia.xlsx -> True
 - educacao_saude.xlsx -> True


## 6. Estatísticas descritivas com `describe()`

Cada linha do `describe()` resume as **5 regiões** (`count = 5`): média, desvio padrão, mínimo, quartis e máximo do número de pessoas, separado por sexo.

In [7]:
print("Curso superior — homens e mulheres por região")
display(superior.describe())

Curso superior — homens e mulheres por região
            Homens     Mulheres
count         5.00         5.00
mean  2,095,706.80 3,075,151.40
std   1,910,348.19 2,577,597.51
min     590,558.00   934,609.00
25%     982,489.00 1,430,013.00
50%   1,744,994.00 2,570,005.00
75%   1,768,855.00 3,003,939.00
max   5,391,638.00 7,437,191.00


In [8]:
print("Ciência, Tecnologia, Engenharias e Matemática — homens e mulheres por região")
display(tecnologia.describe())

Ciência, Tecnologia, Engenharias e Matemática — homens e mulheres por região
            Homens   Mulheres
count         5.00       5.00
mean    550,083.60 279,724.00
std     593,417.87 266,255.48
min     133,055.00  76,601.00
25%     211,124.00 115,834.00
50%     389,993.00 231,395.00
75%     427,372.00 235,543.00
max   1,588,874.00 739,247.00


In [9]:
print("Educação, Serviços pessoais, Saúde e Bem-estar — homens e mulheres por região")
display(educacao_saude.describe())

Educação, Serviços pessoais, Saúde e Bem-estar — homens e mulheres por região
          Homens     Mulheres
count       5.00         5.00
mean  335,596.80 1,280,344.80
std   257,699.41   977,950.63
min   141,169.00   458,616.00
25%   156,048.00   593,101.00
50%   248,574.00 1,007,718.00
75%   363,905.00 1,458,181.00
max   768,288.00 2,884,108.00
